In [1]:
# Cell 1: Build H2 fermionic Hamiltonian and Hermitian fermionic terms

import pandas as pd

from openfermion.chem import MolecularData
from openfermion.ops import FermionOperator
from openfermion.transforms import get_fermion_operator, normal_ordered
from openfermion.utils import hermitian_conjugated
from openfermionpyscf import run_pyscf

pd.set_option("display.max_colwidth", None)


def format_fermion_term(term):
    if term == ():
        return "I"

    pieces = []
    for orbital, action in term:
        if action == 1:
            pieces.append(f"a_{orbital}^dagger")
        else:
            pieces.append(f"a_{orbital}")
    return " ".join(pieces)


def sort_fermion_key(term):
    return (len(term), term)


def coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_fermion_key(item[0])):
        pieces.append(f"{coeff_to_str(coeff, digits)} {format_fermion_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def dagger_term_key(term):
    """
    Return the OpenFermion key for O^dagger, where O is one monomial.
    """
    O = FermionOperator(term, 1.0)
    O_dag = normal_ordered(hermitian_conjugated(O))
    O_dag.compress(abs_tol=1e-12)

    if len(O_dag.terms) != 1:
        raise ValueError(f"Expected one dagger term, got: {O_dag}")

    return next(iter(O_dag.terms.keys()))


def make_hermitian_fermionic_terms(fermion_hamiltonian, tol=1e-12):
    """
    Group raw monomials into Hermitian fermionic Hamiltonian terms.

    If O is self-adjoint, keep c O.
    If O is not self-adjoint, group c O + c* O^dagger.
    """
    used = set()
    hermitian_terms = []

    for term, coeff in fermion_hamiltonian.terms.items():
        if term in used:
            continue

        dag = dagger_term_key(term)

        if dag == term:
            T = FermionOperator(term, coeff)
            used.add(term)
        else:
            dag_coeff = fermion_hamiltonian.terms.get(dag, 0.0)

            T = FermionOperator(term, coeff)
            T += FermionOperator(dag, dag_coeff)

            used.add(term)
            used.add(dag)

        T = normal_ordered(T)
        T.compress(abs_tol=tol)
        hermitian_terms.append(T)

    return hermitian_terms


def build_h10_fermionic_hamiltonian(
    bond_length=0.7414,
    basis="sto-3g",
    multiplicity=1,
    charge=0,
):
    geometry = [
        ("H", (0.0, 0.0, 0.0 * bond_length)),
        ("H", (0.0, 0.0, 1.0 * bond_length)),
        ("H", (0.0, 0.0, 2.0 * bond_length)),
        ("H", (0.0, 0.0, 3.0 * bond_length)),
        ("H", (0.0, 0.0, 4.0 * bond_length)),
        ("H", (0.0, 0.0, 5.0 * bond_length)),
        ("H", (0.0, 0.0, 6.0 * bond_length)),
        ("H", (0.0, 0.0, 7.0 * bond_length)),
        ("H", (0.0, 0.0, 8.0 * bond_length)),
        ("H", (0.0, 0.0, 9.0 * bond_length)),
    ]

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description=f"H10_{bond_length}",
    )

    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=False,
    )

    molecular_hamiltonian = molecule.get_molecular_hamiltonian()
    fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)

    fermion_hamiltonian = normal_ordered(fermion_hamiltonian)
    fermion_hamiltonian.compress(abs_tol=1e-12)

    return molecule, fermion_hamiltonian


molecule, Hf = build_h10_fermionic_hamiltonian(
    bond_length=0.7414,
    basis="sto-3g",
)

hermitian_terms = make_hermitian_fermionic_terms(Hf)

print("Molecule: H10")
print("Basis: STO-3G")
print("Number of electrons:", molecule.n_electrons)
print("Number of spatial orbitals:", molecule.n_orbitals)
print("Number of spin orbitals / fermionic modes:", molecule.n_qubits)
print("Number of raw OpenFermion monomial terms:", len(Hf.terms))
print("Number of Hermitian fermionic terms:", len(hermitian_terms))

print("\n=== Full fermionic Hamiltonian H_f ===")
print(Hf)


# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

raw_rows = []

for idx, (term, coeff) in enumerate(
    sorted(Hf.terms.items(), key=lambda item: sort_fermion_key(item[0]))
):
    raw_rows.append(
        {
            "raw_index": idx,
            "coefficient": coeff_to_str(coeff),
            "monomial": format_fermion_term(term),
            "OpenFermion_key": term,
        }
    )

raw_df = pd.DataFrame(raw_rows)

print("\n=== Raw fermionic monomials c_alpha O_alpha ===")
display(raw_df)


hermitian_rows = []

for i, T in enumerate(hermitian_terms):
    hermitian_rows.append(
        {
            "vertex": f"T_{i}",
            "number_of_monomials": len(T.terms),
            "fermionic_term": operator_to_string(T),
        }
    )

hermitian_df = pd.DataFrame(hermitian_rows)

print("\n=== Hermitian fermionic terms T_i ===")
display(hermitian_df)

Molecule: H10
Basis: STO-3G
Number of electrons: 10
Number of spatial orbitals: 10
Number of spin orbitals / fermionic modes: 20
Number of raw OpenFermion monomial terms: 7151
Number of Hermitian fermionic terms: 3681

=== Full fermionic Hamiltonian H_f ===
13.76808794966473 [] +
-3.5359549988639274 [0^ 0] +
0.1649426220344185 [0^ 4] +
-0.051806732853524017 [0^ 8] +
0.026106398124277494 [0^ 12] +
0.019992844863909835 [0^ 16] +
-0.40765568222464466 [1^ 0^ 1 0] +
-0.13135472084934996 [1^ 0^ 3 2] +
-0.07668556823757515 [1^ 0^ 4 1] +
0.07668556823757515 [1^ 0^ 5 0] +
-0.08917189612324626 [1^ 0^ 5 4] +
0.052053002836877624 [1^ 0^ 6 3] +
-0.052053002836877624 [1^ 0^ 7 2] +
-0.06438293979922091 [1^ 0^ 7 6] +
-0.004077256614803865 [1^ 0^ 8 1] +
0.04112529779298367 [1^ 0^ 8 5] +
0.004077256614803865 [1^ 0^ 9 0] +
-0.04112529779298367 [1^ 0^ 9 4] +
-0.05952844459691587 [1^ 0^ 9 8] +
-0.006256112762920704 [1^ 0^ 10 3] +
-0.032168133955602515 [1^ 0^ 10 7] +
0.006256112762920704 [1^ 0^ 11 2] +
0.03

,raw_index,coefficient,monomial,OpenFermion_key
0,0,+13.76808795,I,()
1,1,-3.53595500,a_0^dagger a_0,"((0, 1), (0, 0))"
2,2,+0.16494262,a_0^dagger a_4,"((0, 1), (4, 0))"
3,3,-0.05180673,a_0^dagger a_8,"((0, 1), (8, 0))"
4,4,+0.02610640,a_0^dagger a_12,"((0, 1), (12, 0))"
...,...,...,...,...
7146,7146,-0.00194184,a_19^dagger a_18^dagger a_19 a_2,"((19, 1), (18, 1), (19, 0), (2, 0))"
7147,7147,-0.00054843,a_19^dagger a_18^dagger a_19 a_6,"((19, 1), (18, 1), (19, 0), (6, 0))"
7148,7148,+0.00061042,a_19^dagger a_18^dagger a_19 a_10,"((19, 1), (18, 1), (19, 0), (10, 0))"
7149,7149,+0.10174657,a_19^dagger a_18^dagger a_19 a_14,"((19, 1), (18, 1), (19, 0), (14, 0))"



=== Hermitian fermionic terms T_i ===


,vertex,number_of_monomials,fermionic_term
0,T_0,1,+13.76808795 I
1,T_1,1,-3.53595500 a_0^dagger a_0
2,T_2,2,+0.16494262 a_0^dagger a_4 + +0.16494262 a_4^dagger a_0
3,T_3,2,-0.05180673 a_0^dagger a_8 + -0.05180673 a_8^dagger a_0
4,T_4,2,+0.02610640 a_0^dagger a_12 + +0.02610640 a_12^dagger a_0
...,...,...,...
3676,T_3676,2,+0.18078400 a_18^dagger a_17^dagger a_19 a_16 + +0.18078400 a_19^dagger a_16^dagger a_18 a_17
3677,T_3677,1,-0.41614532 a_19^dagger a_16^dagger a_19 a_16
3678,T_3678,1,-0.41614532 a_18^dagger a_17^dagger a_18 a_17
3679,T_3679,1,-0.23536132 a_19^dagger a_17^dagger a_19 a_17


In [3]:
# Cell 2 alpha: Faster H2 / molecular fermionic noncommutation graph
#
# Main idea:
#   1. Use cheap fermionic index rules first.
#   2. Only if rules cannot decide, use exact OpenFermion symbolic commutator.
#   3. Never build sparse/dense matrices.
#
# This cell assumes Cell 1 already defined:
#   hermitian_terms
#   operator_to_string

import time
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.utils import commutator
from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Fermionic key utilities
# ------------------------------------------------------------

def key_modes(key):
    """
    Modes appearing in one OpenFermion monomial key.

    Example:
        ((3, 1), (0, 1), (3, 0), (0, 0)) -> {0, 3}
    """
    return frozenset(mode for mode, action in key)


def key_creations(key):
    """
    Creation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 1)


def key_annihilations(key):
    """
    Annihilation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 0)


def key_net_delta(key):
    """
    Net occupation change caused by a monomial.

    creation contributes +1
    annihilation contributes -1

    For example:
        a_2^dagger a_1^dagger a_3 a_0
    has delta:
        +1 on modes 2 and 1
        -1 on modes 3 and 0
    """
    delta = Counter()

    for mode, action in key:
        if action == 1:
            delta[mode] += 1
        else:
            delta[mode] -= 1

    return delta


def is_diagonal_key(key):
    """
    True if a monomial preserves occupation mode-by-mode.

    Examples:
        a_p^dagger a_p is diagonal.
        a_p^dagger a_q^dagger a_q a_p is diagonal.
        a_p^dagger a_q is not diagonal when p != q.
    """
    delta = key_net_delta(key)
    return all(value == 0 for value in delta.values())


def is_even_key(key):
    """
    Electronic Hamiltonian terms normally have even fermionic parity:
    length 0, 2, or 4.
    """
    return len(key) % 2 == 0


# ------------------------------------------------------------
# Safe monomial-level commutation rules
# ------------------------------------------------------------

def diagonal_key_commutes_with_key(diagonal_key, other_key):
    """
    Safe rule:

    A diagonal occupation operator depending on modes S commutes with another
    monomial if the other monomial has zero net occupation change on every
    mode in S.
    """
    support = key_modes(diagonal_key)
    delta = key_net_delta(other_key)

    return all(delta.get(mode, 0) == 0 for mode in support)


def no_cross_contractions_even_commute(key_a, key_b):
    """
    Safe rule for normal-ordered even fermionic monomials.

    If there are no possible cross contractions:
        annihilations(A) intersect creations(B) = empty
        annihilations(B) intersect creations(A) = empty

    then even monomials commute.

    This catches many cases beyond completely disjoint support.
    """
    if not is_even_key(key_a) or not is_even_key(key_b):
        return False

    a_ann = key_annihilations(key_a)
    a_cre = key_creations(key_a)

    b_ann = key_annihilations(key_b)
    b_cre = key_creations(key_b)

    return a_ann.isdisjoint(b_cre) and b_ann.isdisjoint(a_cre)


def monomial_pair_definitely_commutes(key_a, key_b):
    """
    Return (True, reason) only when we are sure two monomials commute.
    Return (False, None) if the rule cannot decide.

    Important:
        False here does NOT mean noncommuting.
        It only means "unknown; use exact symbolic fallback."
    """
    # Identity commutes with everything.
    if key_a == () or key_b == ():
        return True, "identity"

    # Any monomial commutes with itself.
    if key_a == key_b:
        return True, "same_monomial"

    # Diagonal occupation-like monomials commute with each other.
    if is_diagonal_key(key_a) and is_diagonal_key(key_b):
        return True, "diagonal_diagonal"

    # Diagonal with excitation-like term, if excitation preserves diagonal support.
    if is_diagonal_key(key_a) and diagonal_key_commutes_with_key(key_a, key_b):
        return True, "diagonal_support_preserved"

    if is_diagonal_key(key_b) and diagonal_key_commutes_with_key(key_b, key_a):
        return True, "diagonal_support_preserved"

    # Even monomials with no cross contractions commute.
    if no_cross_contractions_even_commute(key_a, key_b):
        return True, "no_cross_contractions_even"

    return False, None


# ------------------------------------------------------------
# Operator-level metadata and precheck
# ------------------------------------------------------------

def operator_metadata(op):
    """
    Precompute simple structural data for one FermionOperator.
    """
    keys = list(op.terms.keys())

    modes = set()
    for key in keys:
        modes.update(key_modes(key))

    return {
        "is_zero": len(keys) == 0,
        "only_identity": len(keys) == 1 and keys[0] == (),
        "modes": frozenset(modes),
        "is_even": all(is_even_key(key) for key in keys),
        "is_diagonal": all(is_diagonal_key(key) for key in keys),
        "number_of_monomials": len(keys),
    }


def operator_pair_definitely_commutes(A, B, meta_A, meta_B):
    """
    Return (True, reason) only for guaranteed-commuting pairs.
    Return (False, None) when unresolved.
    """
    if meta_A["is_zero"] or meta_B["is_zero"]:
        return True, "zero"

    if meta_A["only_identity"] or meta_B["only_identity"]:
        return True, "identity"

    # Very cheap global rule:
    # disjoint even fermionic operators commute.
    if (
        meta_A["is_even"]
        and meta_B["is_even"]
        and meta_A["modes"].isdisjoint(meta_B["modes"])
    ):
        return True, "disjoint_even_support"

    # Diagonal occupation-like operators commute with each other.
    if meta_A["is_diagonal"] and meta_B["is_diagonal"]:
        return True, "diagonal_diagonal"

    # More detailed but still cheap:
    # if every monomial pair has a safe commuting reason, the sums commute.
    reasons = Counter()

    for key_a in A.terms:
        for key_b in B.terms:
            ok, reason = monomial_pair_definitely_commutes(key_a, key_b)

            if not ok:
                return False, None

            reasons[reason] += 1

    if len(reasons) > 0:
        main_reason = reasons.most_common(1)[0][0]
        return True, f"all_monomial_pairs_{main_reason}"

    return False, None


# ------------------------------------------------------------
# Exact symbolic fallback
# ------------------------------------------------------------

def exact_symbolic_fermionic_commutator(A, B, tol=1e-12):
    """
    Exact symbolic commutator in fermionic algebra.

    This does not build a 2^n matrix.
    """
    C = normal_ordered(commutator(A, B))
    C.compress(abs_tol=tol)
    return C


# ------------------------------------------------------------
# Alpha graph builder
# ------------------------------------------------------------

def build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
):
    """
    Build noncommutation graph using:
        fast safe index rules first,
        exact symbolic OpenFermion fallback only when needed.

    Parameters
    ----------
    hermitian_terms:
        list of FermionOperator terms T_i from Cell 1.

    tol:
        numerical compression tolerance.

    store_commutators:
        False is recommended for large molecules.
        True is useful for H2 debugging, but can be memory-heavy.

    Returns
    -------
    G:
        networkx.Graph

    stats_df:
        pandas.DataFrame with timing and skip counts
    """
    t_start = time.perf_counter()

    G = nx.Graph()
    stats = Counter()

    metadata = [operator_metadata(T) for T in hermitian_terms]

    # Add vertices.
    for i, T in enumerate(hermitian_terms):
        G.add_node(
            i,
            label=f"T_{i}",
            operator=T,
            operator_string=operator_to_string(T),
            number_of_monomials=len(T.terms),
            modes=sorted(metadata[i]["modes"]),
            is_diagonal=metadata[i]["is_diagonal"],
            is_even=metadata[i]["is_even"],
        )

    n = len(hermitian_terms)

    # Pairwise graph construction.
    for i in range(n):
        A = hermitian_terms[i]
        meta_A = metadata[i]

        for j in range(i + 1, n):
            B = hermitian_terms[j]
            meta_B = metadata[j]

            stats["total_pairs"] += 1

            # 1. Fast guaranteed-commuting rules.
            definitely_commutes, reason = operator_pair_definitely_commutes(
                A, B, meta_A, meta_B
            )

            if definitely_commutes:
                stats["pairs_skipped_by_index_rules"] += 1
                stats[f"skip_{reason}"] += 1
                continue

            # 2. Exact symbolic fallback.
            stats["pairs_sent_to_exact_symbolic"] += 1

            Cij = exact_symbolic_fermionic_commutator(A, B, tol=tol)

            if len(Cij.terms) != 0:
                stats["noncommuting_edges"] += 1

                edge_data = {
                    "method": "exact_symbolic_fallback",
                    "meaning": f"[T_{i}, T_{j}] != 0",
                }

                if store_commutators:
                    edge_data["commutator"] = Cij
                    edge_data["commutator_string"] = operator_to_string(Cij)
                else:
                    edge_data["commutator_string"] = (
                        "(not stored; rerun with store_commutators=True)"
                    )

                G.add_edge(i, j, **edge_data)

            else:
                stats["exact_symbolic_found_commuting"] += 1

    elapsed = time.perf_counter() - t_start

    stats["vertices"] = n
    stats["edges"] = G.number_of_edges()
    stats["commuting_pairs"] = stats["total_pairs"] - G.number_of_edges()
    stats["elapsed_seconds"] = elapsed

    stats_df = pd.DataFrame([dict(stats)])

    return G, stats_df


# ------------------------------------------------------------
# Run alpha graph builder
# ------------------------------------------------------------

# For H2 debugging, you can set store_commutators=True.
# For larger molecules, keep this False.
G_alpha, stats_df = build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
)

print("=== Fermionic noncommutation graph alpha summary ===")
display(stats_df)

print("Number of vertices / fermionic terms:", G_alpha.number_of_nodes())
print("Number of noncommuting pairs / edges:", G_alpha.number_of_edges())
print("Is graph bipartite?", nx.is_bipartite(G_alpha))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G_alpha.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G_alpha.degree[i],
            "commutes_with_all": G_alpha.degree[i] == 0,
            "number_of_monomials": data["number_of_monomials"],
            "modes": data["modes"],
            "is_diagonal": data["is_diagonal"],
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df_alpha = pd.DataFrame(vertex_rows)

print("\n=== Alpha vertices: fermionic terms ===")
display(vertex_df_alpha)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G_alpha.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": data["meaning"],
            "method": data["method"],
            "commutator": data["commutator_string"],
        }
    )

edge_df_alpha = pd.DataFrame(edge_rows)

print("\n=== Alpha edges: noncommuting pairs ===")
display(edge_df_alpha)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G_alpha)

# node_labels = {
#     i: f"T_{i}"
#     for i in G_alpha.nodes()
# }

# node_sizes = [
#     1000 + 250 * G_alpha.degree[i]
#     for i in G_alpha.nodes()
# ]

# nx.draw_networkx_nodes(G_alpha, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G_alpha, pos, width=1.5)
# nx.draw_networkx_labels(
#     G_alpha,
#     pos,
#     labels=node_labels,
#     font_size=11,
#     font_weight="bold",
# )

# plt.title("Fermionic Noncommutation Graph Alpha")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph alpha summary ===


,total_pairs,pairs_skipped_by_index_rules,skip_identity,pairs_sent_to_exact_symbolic,noncommuting_edges,skip_disjoint_even_support,skip_diagonal_diagonal,skip_all_monomial_pairs_diagonal_support_preserved,exact_symbolic_found_commuting,vertices,edges,commuting_pairs,elapsed_seconds
0,6773040,3112975,3680,3660065,3379643,3092535,3800,12960,280422,3681,3379643,3393397,208.680862


Number of vertices / fermionic terms: 3681
Number of noncommuting pairs / edges: 3379643
Is graph bipartite? False

=== Alpha vertices: fermionic terms ===


,vertex,degree,commutes_with_all,number_of_monomials,modes,is_diagonal,fermionic_term
0,T_0,0,True,1,[],True,+13.76808795 I
1,T_1,618,False,1,[0],True,-3.53595500 a_0^dagger a_0
2,T_2,1210,False,2,"[0, 4]",False,+0.16494262 a_0^dagger a_4 + +0.16494262 a_4^dagger a_0
3,T_3,1210,False,2,"[0, 8]",False,-0.05180673 a_0^dagger a_8 + -0.05180673 a_8^dagger a_0
4,T_4,1210,False,2,"[0, 12]",False,+0.02610640 a_0^dagger a_12 + +0.02610640 a_12^dagger a_0
...,...,...,...,...,...,...,...
3676,T_3676,1974,False,2,"[16, 17, 18, 19]",False,+0.18078400 a_18^dagger a_17^dagger a_19 a_16 + +0.18078400 a_19^dagger a_16^dagger a_18 a_17
3677,T_3677,1113,False,1,"[16, 19]",True,-0.41614532 a_19^dagger a_16^dagger a_19 a_16
3678,T_3678,1113,False,1,"[17, 18]",True,-0.41614532 a_18^dagger a_17^dagger a_18 a_17
3679,T_3679,1056,False,1,"[17, 19]",True,-0.23536132 a_19^dagger a_17^dagger a_19 a_17



=== Alpha edges: noncommuting pairs ===


,source,target,meaning,method,commutator
0,T_1,T_2,"[T_1, T_2] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1,T_1,T_3,"[T_1, T_3] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
2,T_1,T_4,"[T_1, T_4] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3,T_1,T_5,"[T_1, T_5] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
4,T_1,T_62,"[T_1, T_62] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
...,...,...,...,...,...
3379638,T_3672,T_3676,"[T_3672, T_3676] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3379639,T_3673,T_3674,"[T_3673, T_3674] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3379640,T_3674,T_3680,"[T_3674, T_3680] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3379641,T_3676,T_3677,"[T_3676, T_3677] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)


In [4]:
# Cell 2: Build the H2 fermionic noncommutation graph

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


def fermionic_commutator(A, B, tol=1e-12):
    """
    Compute [A, B] = AB - BA directly in the fermionic algebra.
    """
    C = normal_ordered(A * B - B * A)
    C.compress(abs_tol=tol)
    return C


def commute(A, B, tol=1e-12):
    """
    Return True if [A, B] = 0.
    """
    C = fermionic_commutator(A, B, tol=tol)
    return len(C.terms) == 0


# ------------------------------------------------------------
# Build noncommutation graph
# ------------------------------------------------------------
# Vertex i = Hermitian fermionic term T_i
# Edge (i, j) exists if [T_i, T_j] != 0

G = nx.Graph()

for i, T in enumerate(hermitian_terms):
    G.add_node(
        i,
        label=f"T_{i}",
        operator=T,
        operator_string=operator_to_string(T),
        number_of_monomials=len(T.terms),
    )

for i in range(len(hermitian_terms)):
    for j in range(i + 1, len(hermitian_terms)):
        Cij = fermionic_commutator(hermitian_terms[i], hermitian_terms[j])

        if len(Cij.terms) != 0:
            G.add_edge(
                i,
                j,
                commutator=Cij,
                commutator_string=operator_to_string(Cij),
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_vertices = G.number_of_nodes()
n_total_pairs = n_vertices * (n_vertices - 1) // 2
n_noncommuting_pairs = G.number_of_edges()
n_commuting_pairs = n_total_pairs - n_noncommuting_pairs

print("=== Fermionic noncommutation graph summary ===")
print("Number of vertices / fermionic terms:", n_vertices)
print("Number of total unordered pairs:", n_total_pairs)
print("Number of noncommuting pairs / edges:", n_noncommuting_pairs)
print("Number of commuting pairs:", n_commuting_pairs)
print("Is graph bipartite?", nx.is_bipartite(G))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G.degree[i],
            "commutes_with_all": G.degree[i] == 0,
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df = pd.DataFrame(vertex_rows)

print("\n=== Vertices: fermionic terms ===")
display(vertex_df)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": f"[T_{i}, T_{j}] != 0",
            "commutator": data["commutator_string"],
        }
    )

edge_df = pd.DataFrame(edge_rows)

print("\n=== Edges: noncommuting pairs ===")
display(edge_df)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G)

# node_labels = {
#     i: f"T_{i}"
#     for i in G.nodes()
# }

# node_sizes = [
#     1000 + 250 * G.degree[i]
#     for i in G.nodes()
# ]

# nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G, pos, width=1.5)
# nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight="bold")

# plt.title("H2 Fermionic Noncommutation Graph")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph summary ===
Number of vertices / fermionic terms: 3681
Number of total unordered pairs: 6773040
Number of noncommuting pairs / edges: 3379643
Number of commuting pairs: 3393397
Is graph bipartite? False

=== Vertices: fermionic terms ===


,vertex,degree,commutes_with_all,fermionic_term
0,T_0,0,True,+13.76808795 I
1,T_1,618,False,-3.53595500 a_0^dagger a_0
2,T_2,1210,False,+0.16494262 a_0^dagger a_4 + +0.16494262 a_4^dagger a_0
3,T_3,1210,False,-0.05180673 a_0^dagger a_8 + -0.05180673 a_8^dagger a_0
4,T_4,1210,False,+0.02610640 a_0^dagger a_12 + +0.02610640 a_12^dagger a_0
...,...,...,...,...
3676,T_3676,1974,False,+0.18078400 a_18^dagger a_17^dagger a_19 a_16 + +0.18078400 a_19^dagger a_16^dagger a_18 a_17
3677,T_3677,1113,False,-0.41614532 a_19^dagger a_16^dagger a_19 a_16
3678,T_3678,1113,False,-0.41614532 a_18^dagger a_17^dagger a_18 a_17
3679,T_3679,1056,False,-0.23536132 a_19^dagger a_17^dagger a_19 a_17



=== Edges: noncommuting pairs ===


,source,target,meaning,commutator
0,T_1,T_2,"[T_1, T_2] != 0",-0.58322969 a_0^dagger a_4 + +0.58322969 a_4^dagger a_0
1,T_1,T_3,"[T_1, T_3] != 0",+0.18318628 a_0^dagger a_8 + -0.18318628 a_8^dagger a_0
2,T_1,T_4,"[T_1, T_4] != 0",-0.09231105 a_0^dagger a_12 + +0.09231105 a_12^dagger a_0
3,T_1,T_5,"[T_1, T_5] != 0",-0.07069380 a_0^dagger a_16 + +0.07069380 a_16^dagger a_0
4,T_1,T_62,"[T_1, T_62] != 0",+0.27115672 a_1^dagger a_0^dagger a_4 a_1 + -0.27115672 a_4^dagger a_1^dagger a_1 a_0
...,...,...,...,...
3379638,T_3672,T_3676,"[T_3672, T_3676] != 0",-0.05024987 a_18^dagger a_17^dagger a_15^dagger a_19 a_16 a_15 + +0.05024987 a_19^dagger a_16^dagger a_15^dagger a_18 a_17 a_15
3379639,T_3673,T_3674,"[T_3673, T_3674] != 0",-0.07745133 a_17^dagger a_16^dagger a_19 a_18 + +0.07745133 a_19^dagger a_18^dagger a_17 a_16
3379640,T_3674,T_3680,"[T_3674, T_3680] != 0",-0.09527916 a_17^dagger a_16^dagger a_19 a_18 + +0.09527916 a_19^dagger a_18^dagger a_17 a_16
3379641,T_3676,T_3677,"[T_3676, T_3677] != 0",+0.07523242 a_18^dagger a_17^dagger a_19 a_16 + -0.07523242 a_19^dagger a_16^dagger a_18 a_17


In [5]:
print("Same edge set?")
print(set(G.edges()) == set(G_alpha.edges()))

Same edge set?
True


In [6]:
# Cell 3: Color graph, build commuting blocks, map JW/BK, verify commutation

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from openfermion.ops import FermionOperator
from openfermion.transforms import normal_ordered, jordan_wigner, bravyi_kitaev

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# 1. Color the noncommutation graph
# ------------------------------------------------------------
# Since edges mean noncommutation, each color class is a commuting group.

coloring = nx.coloring.greedy_color(G, strategy="largest_first")

color_groups = {}

for node, color in coloring.items():
    color_groups.setdefault(color, []).append(node)

for color in color_groups:
    color_groups[color] = sorted(color_groups[color])

color_names = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "gray",
]

color_name = {
    color: color_names[color] if color < len(color_names) else f"color_{color}"
    for color in color_groups
}

num_grouped_terms = sum(len(nodes) for nodes in color_groups.values())

print("Number of fermionic terms / vertices:", G.number_of_nodes())
print("Number of colors / commuting groups:", len(color_groups))
print("Number of grouped terms:", num_grouped_terms)

assert num_grouped_terms == G.number_of_nodes()


# ------------------------------------------------------------
# 2. Verify each color group is mutually commuting
# ------------------------------------------------------------

def verify_commuting_group(nodes, tol=1e-12):
    for i, j in combinations(nodes, 2):
        A = G.nodes[i]["operator"]
        B = G.nodes[j]["operator"]

        if not commute(A, B, tol=tol):
            return False

    return True


group_summary_rows = []

for color, nodes in sorted(color_groups.items()):
    group_summary_rows.append(
        {
            "color_id": color,
            "color_name": color_name[color],
            "number_of_terms": len(nodes),
            "vertices": [f"T_{i}" for i in nodes],
            "verified_mutually_commuting": verify_commuting_group(nodes),
        }
    )

group_summary_df = pd.DataFrame(group_summary_rows)

print("\n=== Commuting groups from graph coloring ===")
display(group_summary_df)


# ------------------------------------------------------------
# 3. Build Hamiltonian pieces by color
# ------------------------------------------------------------

H_by_color = {}

for color, nodes in sorted(color_groups.items()):
    H_color = FermionOperator.zero()

    for node in nodes:
        H_color += G.nodes[node]["operator"]

    H_color = normal_ordered(H_color)
    H_color.compress(abs_tol=1e-12)

    H_by_color[color] = H_color


color_block_rows = []

for color, nodes in sorted(color_groups.items()):
    for local_index, node in enumerate(nodes, start=1):
        color_block_rows.append(
            {
                "color_block": f"H_{color_name[color]}",
                "local_term_name": f"{color_name[color][0].upper()}_{local_index}",
                "vertex": f"T_{node}",
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

color_block_df = pd.DataFrame(color_block_rows)

print("\n=== Which T_i belongs to which color block ===")
display(color_block_df)


print("\n=== Hamiltonian split by commuting color groups ===")

for color, H_color in H_by_color.items():
    nodes = color_groups[color]

    print("\n" + "=" * 80)
    print(f"H_{color_name[color]} consists of:")
    print(" + ".join([f"T_{node}" for node in nodes]))

    print(f"\nSummed operator H_{color_name[color]} =")
    print(H_color)


# ------------------------------------------------------------
# 4. Trotter ordering induced by color groups
# ------------------------------------------------------------

trotter_order = []

for color, nodes in sorted(color_groups.items()):
    for node in nodes:
        trotter_order.append(node)

print("\n=== Trotter order by commuting color groups ===")
print([f"T_{i}" for i in trotter_order])


# ------------------------------------------------------------
# 5. Helper functions for JW/BK output
# ------------------------------------------------------------

def sort_qubit_key(term):
    return (len(term), term)


def format_qubit_term(term):
    if term == ():
        return "I"

    return " ".join([f"{pauli}{qubit}" for qubit, pauli in term])


def qubit_coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def qubit_operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_qubit_key(item[0])):
        pieces.append(f"{qubit_coeff_to_str(coeff, digits)} {format_qubit_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def apply_bk(op, n_qubits):
    try:
        return bravyi_kitaev(op, n_qubits=n_qubits)
    except TypeError:
        return bravyi_kitaev(op, n_qubits)


def qubit_commutator(A, B, tol=1e-12):
    C = A * B - B * A
    C.compress(abs_tol=tol)
    return C


def qubit_commute(A, B, tol=1e-12):
    C = qubit_commutator(A, B, tol=tol)
    return len(C.terms) == 0


n_qubits = molecule.n_qubits

print("\nNumber of qubits / spin orbitals:", n_qubits)


# ------------------------------------------------------------
# 6. Map each fermionic vertex T_i to JW(T_i) and BK(T_i)
# ------------------------------------------------------------

mapped_rows = []

for node in sorted(G.nodes()):
    T_i = G.nodes[node]["operator"]

    JW_T_i = jordan_wigner(T_i)
    JW_T_i.compress(abs_tol=1e-12)

    BK_T_i = apply_bk(T_i, n_qubits=n_qubits)
    BK_T_i.compress(abs_tol=1e-12)

    G.nodes[node]["JW_operator"] = JW_T_i
    G.nodes[node]["BK_operator"] = BK_T_i
    G.nodes[node]["JW_operator_string"] = qubit_operator_to_string(JW_T_i)
    G.nodes[node]["BK_operator_string"] = qubit_operator_to_string(BK_T_i)

    node_color = coloring[node]
    node_color_name = color_name[node_color]

    mapped_rows.append(
        {
            "color": node_color_name,
            "vertex": f"T_{node}",
            "fermionic_term": G.nodes[node]["operator_string"],
            "number_of_JW_Pauli_strings": len(JW_T_i.terms),
            "JW_transform": qubit_operator_to_string(JW_T_i),
            "number_of_BK_Pauli_strings": len(BK_T_i.terms),
            "BK_transform": qubit_operator_to_string(BK_T_i),
        }
    )

mapped_terms_df = pd.DataFrame(mapped_rows)

print("\n=== Fermionic terms mapped to JW and BK ===")
display(mapped_terms_df)


# ------------------------------------------------------------
# 7. Map each color block H_color to JW and BK
# ------------------------------------------------------------

JW_by_color = {}
BK_by_color = {}

block_rows = []

for color, H_color in sorted(H_by_color.items()):
    JW_color = jordan_wigner(H_color)
    JW_color.compress(abs_tol=1e-12)

    BK_color = apply_bk(H_color, n_qubits=n_qubits)
    BK_color.compress(abs_tol=1e-12)

    JW_by_color[color] = JW_color
    BK_by_color[color] = BK_color

    block_rows.append(
        {
            "color_block": f"H_{color_name[color]}",
            "fermionic_vertices": " + ".join([f"T_{node}" for node in color_groups[color]]),
            "number_of_fermionic_terms": len(color_groups[color]),
            "number_of_JW_Pauli_strings": len(JW_color.terms),
            "JW_block": qubit_operator_to_string(JW_color),
            "number_of_BK_Pauli_strings": len(BK_color.terms),
            "BK_block": qubit_operator_to_string(BK_color),
        }
    )

block_map_df = pd.DataFrame(block_rows)

print("\n=== Color blocks mapped to JW and BK ===")
display(block_map_df)


# ------------------------------------------------------------
# 8. Verify same-color terms commute after JW and BK
# ------------------------------------------------------------

verification_rows = []

for color, nodes in sorted(color_groups.items()):
    for i, j in combinations(nodes, 2):
        Ti = G.nodes[i]["operator"]
        Tj = G.nodes[j]["operator"]

        JW_Ti = G.nodes[i]["JW_operator"]
        JW_Tj = G.nodes[j]["JW_operator"]

        BK_Ti = G.nodes[i]["BK_operator"]
        BK_Tj = G.nodes[j]["BK_operator"]

        verification_rows.append(
            {
                "color_group": color_name[color],
                "pair": f"T_{i}, T_{j}",
                "fermionic_commute": commute(Ti, Tj),
                "JW_commute": qubit_commute(JW_Ti, JW_Tj),
                "BK_commute": qubit_commute(BK_Ti, BK_Tj),
            }
        )

verification_df = pd.DataFrame(verification_rows)

print("\n=== Verify commuting groups after JW/BK mapping ===")
display(verification_df)

Number of fermionic terms / vertices: 3681
Number of colors / commuting groups: 277
Number of grouped terms: 3681

=== Commuting groups from graph coloring ===


,color_id,color_name,number_of_terms,vertices,verified_mutually_commuting
0,0,red,17,"[T_0, T_72, T_190, T_206, T_849, T_1318, T_1425, T_3214, T_3254, T_3258, T_3385, T_3456, T_3485, T_3674, T_3675, T_3676, T_3679]",True
1,1,blue,20,"[T_57, T_58, T_59, T_60, T_73, T_207, T_332, T_861, T_1319, T_1426, T_1891, T_2306, T_2369, T_2385, T_2895, T_2953, T_3673, T_3677, T_3678, T_3680]",True
2,2,green,12,"[T_82, T_192, T_347, T_980, T_1320, T_1427, T_2542, T_2617, T_2692, T_2874, T_3244, T_3286]",True
3,3,orange,15,"[T_83, T_334, T_348, T_992, T_1321, T_1428, T_1886, T_2215, T_2368, T_2378, T_2894, T_2952, T_3653, T_3658, T_3661]",True
4,4,purple,12,"[T_84, T_349, T_468, T_981, T_1327, T_1538, T_2214, T_2285, T_2377, T_2688, T_3075, T_3128]",True
...,...,...,...,...,...
272,272,color_272,5,"[T_1745, T_3179, T_3428, T_3593, T_3614]",True
273,273,color_273,4,"[T_482, T_2873, T_3180, T_3360]",True
274,274,color_274,5,"[T_2459, T_2606, T_2945, T_3409, T_3556]",True
275,275,color_275,5,"[T_991, T_2196, T_3007, T_3256, T_3327]",True



=== Which T_i belongs to which color block ===


,color_block,local_term_name,vertex,fermionic_term
0,H_red,R_1,T_0,+13.76808795 I
1,H_red,R_2,T_72,-0.08917190 a_1^dagger a_0^dagger a_5 a_4 + -0.08917190 a_5^dagger a_4^dagger a_1 a_0
2,H_red,R_3,T_190,+0.06346349 a_4^dagger a_0^dagger a_6 a_2 + +0.06346349 a_6^dagger a_2^dagger a_4 a_0
3,H_red,R_4,T_206,+0.08917190 a_4^dagger a_1^dagger a_5 a_0 + +0.08917190 a_5^dagger a_0^dagger a_4 a_1
4,H_red,R_5,T_849,+0.06346349 a_5^dagger a_1^dagger a_7 a_3 + +0.06346349 a_7^dagger a_3^dagger a_5 a_1
...,...,...,...,...
3676,H_color_275,C_3,T_3007,+0.00451985 a_18^dagger a_6^dagger a_18 a_14 + +0.00451985 a_18^dagger a_14^dagger a_18 a_6
3677,H_color_275,C_4,T_3256,+0.06584760 a_12^dagger a_8^dagger a_16 a_12 + +0.06584760 a_16^dagger a_12^dagger a_12 a_8
3678,H_color_275,C_5,T_3327,+0.00022904 a_18^dagger a_8^dagger a_18 a_16 + +0.00022904 a_18^dagger a_16^dagger a_18 a_8
3679,H_color_276,C_1,T_3303,-0.02327134 a_16^dagger a_8^dagger a_16 a_12 + -0.02327134 a_16^dagger a_12^dagger a_16 a_8



=== Hamiltonian split by commuting color groups ===

H_red consists of:
T_0 + T_72 + T_190 + T_206 + T_849 + T_1318 + T_1425 + T_3214 + T_3254 + T_3258 + T_3385 + T_3456 + T_3485 + T_3674 + T_3675 + T_3676 + T_3679

Summed operator H_red =
13.76808794966473 [] +
-0.08917189612324626 [1^ 0^ 5 4] +
-0.08234631449163567 [3^ 2^ 7 6] +
0.06346348937483284 [4^ 0^ 6 2] +
0.08917189612324626 [4^ 1^ 5 0] +
0.08917189612324626 [5^ 0^ 4 1] +
0.06346348937483284 [5^ 1^ 7 3] +
-0.08917189612324626 [5^ 4^ 1 0] +
0.06346348937483284 [6^ 2^ 4 0] +
0.08234631449163567 [6^ 3^ 7 2] +
0.08234631449163567 [7^ 2^ 6 3] +
0.06346348937483284 [7^ 3^ 5 1] +
-0.08234631449163567 [7^ 6^ 3 2] +
-0.07488299483703216 [9^ 8^ 13 12] +
-0.08664522034359738 [11^ 10^ 15 14] +
0.08888469474343709 [12^ 8^ 14 10] +
0.07488299483703216 [12^ 9^ 13 8] +
0.07488299483703216 [13^ 8^ 12 9] +
0.08888469474343709 [13^ 9^ 15 11] +
-0.07488299483703216 [13^ 12^ 9 8] +
0.08888469474343709 [14^ 10^ 12 8] +
0.08664522034359738 [14^ 11^

,color,vertex,fermionic_term,number_of_JW_Pauli_strings,JW_transform,number_of_BK_Pauli_strings,BK_transform
0,red,T_0,+13.76808795 I,1,+13.76808795 I,1,+13.76808795 I
1,color_87,T_1,-3.53595500 a_0^dagger a_0,2,-1.76797750 I + +1.76797750 Z0,2,-1.76797750 I + +1.76797750 Z0
2,color_222,T_2,+0.16494262 a_0^dagger a_4 + +0.16494262 a_4^dagger a_0,2,+0.08247131 X0 Z1 Z2 Z3 X4 + +0.08247131 Y0 Z1 Z2 Z3 Y4,2,+0.08247131 X0 X1 Y3 Y4 X5 + -0.08247131 Y0 X1 Y3 X4 X5
3,color_93,T_3,-0.05180673 a_0^dagger a_8 + -0.05180673 a_8^dagger a_0,2,-0.02590337 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 X8 + -0.02590337 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Y8,2,-0.02590337 X0 X1 X3 Y7 Y8 X9 X11 + +0.02590337 Y0 X1 X3 Y7 X8 X9 X11
4,color_212,T_4,+0.02610640 a_0^dagger a_12 + +0.02610640 a_12^dagger a_0,2,+0.01305320 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + +0.01305320 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12,2,+0.01305320 X0 X1 X3 Y7 Z11 Y12 X13 + -0.01305320 Y0 X1 X3 Y7 Z11 X12 X13
...,...,...,...,...,...,...,...
3676,red,T_3676,+0.18078400 a_18^dagger a_17^dagger a_19 a_16 + +0.18078400 a_19^dagger a_16^dagger a_18 a_17,8,-0.02259800 X16 X17 X18 X19 + -0.02259800 X16 X17 Y18 Y19 + -0.02259800 X16 Y17 X18 Y19 + +0.02259800 X16 Y17 Y18 X19 + +0.02259800 Y16 X17 X18 Y19 + -0.02259800 Y16 X17 Y18 X19 + -0.02259800 Y16 Y17 X18 X19 + -0.02259800 Y16 Y17 Y18 Y19,8,-0.02259800 X16 X18 + -0.02259800 Y16 Y18 + +0.02259800 X16 Z17 X18 + -0.02259800 X16 X18 Z19 + +0.02259800 Y16 Z17 Y18 + -0.02259800 Y16 Y18 Z19 + +0.02259800 X16 Z17 X18 Z19 + +0.02259800 Y16 Z17 Y18 Z19
3677,blue,T_3677,-0.41614532 a_19^dagger a_16^dagger a_19 a_16,4,+0.10403633 I + -0.10403633 Z16 + -0.10403633 Z19 + +0.10403633 Z16 Z19,4,+0.10403633 I + -0.10403633 Z16 + -0.10403633 Z17 Z18 Z19 + +0.10403633 Z16 Z17 Z18 Z19
3678,blue,T_3678,-0.41614532 a_18^dagger a_17^dagger a_18 a_17,4,+0.10403633 I + -0.10403633 Z17 + -0.10403633 Z18 + +0.10403633 Z17 Z18,4,+0.10403633 I + -0.10403633 Z18 + -0.10403633 Z16 Z17 + +0.10403633 Z16 Z17 Z18
3679,red,T_3679,-0.23536132 a_19^dagger a_17^dagger a_19 a_17,4,+0.05884033 I + -0.05884033 Z17 + -0.05884033 Z19 + +0.05884033 Z17 Z19,4,+0.05884033 I + -0.05884033 Z16 Z17 + +0.05884033 Z16 Z18 Z19 + -0.05884033 Z17 Z18 Z19



=== Color blocks mapped to JW and BK ===


,color_block,fermionic_vertices,number_of_fermionic_terms,number_of_JW_Pauli_strings,JW_block,number_of_BK_Pauli_strings,BK_block
0,H_red,T_0 + T_72 + T_190 + T_206 + T_849 + T_1318 + T_1425 + T_3214 + T_3254 + T_3258 + T_3385 + T_3456 + T_3485 + T_3674 + T_3675 + T_3676 + T_3679,17,59,+13.88576861 I + -0.05884033 Z16 + -0.05884033 Z17 + -0.05884033 Z18 + -0.05884033 Z19 + +0.05884033 Z16 Z18 + +0.05884033 Z17 Z19 + -0.02229297 X0 X1 Y4 Y5 + +0.02229297 X0 Y1 Y4 X5 + +0.02229297 Y0 X1 X4 Y5 + -0.02229297 Y0 Y1 X4 X5 + -0.02058658 X2 X3 Y6 Y7 + +0.02058658 X2 Y3 Y6 X7 + +0.02058658 Y2 X3 X6 Y7 + -0.02058658 Y2 Y3 X6 X7 + -0.01872075 X8 X9 Y12 Y13 + +0.01872075 X8 Y9 Y12 X13 + +0.01872075 Y8 X9 X12 Y13 + -0.01872075 Y8 Y9 X12 X13 + -0.02166131 X10 X11 Y14 Y15 + +0.02166131 X10 Y11 Y14 X15 + +0.02166131 Y10 X11 X14 Y15 + -0.02166131 Y10 Y11 X14 X15 + -0.04519600 X16 X17 Y18 Y19 + +0.04519600 X16 Y17 Y18 X19 + +0.04519600 Y16 X17 X18 Y19 + -0.04519600 Y16 Y17 X18 X19 + -0.00793294 X0 Z1 X2 X4 Z5 X6 + -0.00793294 X0 Z1 X2 Y4 Z5 Y6 + +0.00793294 X0 Z1 Y2 X4 Z5 Y6 + -0.00793294 X0 Z1 Y2 Y4 Z5 X6 + -0.00793294 Y0 Z1 X2 X4 Z5 Y6 + +0.00793294 Y0 Z1 X2 Y4 Z5 X6 + -0.00793294 Y0 Z1 Y2 X4 Z5 X6 + -0.00793294 Y0 Z1 Y2 Y4 Z5 Y6 + -0.00793294 X1 Z2 X3 X5 Z6 X7 + -0.00793294 X1 Z2 X3 Y5 Z6 Y7 + +0.00793294 X1 Z2 Y3 X5 Z6 Y7 + -0.00793294 X1 Z2 Y3 Y5 Z6 X7 + -0.00793294 Y1 Z2 X3 X5 Z6 Y7 + +0.00793294 Y1 Z2 X3 Y5 Z6 X7 + -0.00793294 Y1 Z2 Y3 X5 Z6 X7 + -0.00793294 Y1 Z2 Y3 Y5 Z6 Y7 + -0.01111059 X8 Z9 X10 X12 Z13 X14 + -0.01111059 X8 Z9 X10 Y12 Z13 Y14 + +0.01111059 X8 Z9 Y10 X12 Z13 Y14 + -0.01111059 X8 Z9 Y10 Y12 Z13 X14 + -0.01111059 Y8 Z9 X10 X12 Z13 Y14 + +0.01111059 Y8 Z9 X10 Y12 Z13 X14 + -0.01111059 Y8 Z9 Y10 X12 Z13 X14 + -0.01111059 Y8 Z9 Y10 Y12 Z13 Y14 + -0.01111059 X9 Z10 X11 X13 Z14 X15 + -0.01111059 X9 Z10 X11 Y13 Z14 Y15 + +0.01111059 X9 Z10 Y11 X13 Z14 Y15 + -0.01111059 X9 Z10 Y11 Y13 Z14 X15 + -0.01111059 Y9 Z10 X11 X13 Z14 Y15 + +0.01111059 Y9 Z10 X11 Y13 Z14 X15 + -0.01111059 Y9 Z10 Y11 X13 Z14 X15 + -0.01111059 Y9 Z10 Y11 Y13 Z14 Y15,59,+13.88576861 I + -0.05884033 Z16 + -0.05884033 Z18 + -0.05884033 Z16 Z17 + +0.05884033 Z16 Z18 + +0.02229297 X0 Z1 X4 + +0.02229297 X0 X4 Z5 + +0.02229297 Y0 Z1 Y4 + +0.02229297 Y0 Y4 Z5 + +0.00793294 Y1 Y5 Z7 + +0.01872075 X8 Z9 X12 + +0.01872075 X8 X12 Z13 + +0.01872075 Y8 Z9 Y12 + +0.01872075 Y8 Y12 Z13 + +0.04519600 X16 Z17 X18 + +0.04519600 Y16 Z17 Y18 + +0.05884033 Z16 Z18 Z19 + -0.05884033 Z17 Z18 Z19 + -0.00793294 X1 Z2 X5 Z6 + +0.02058658 Z1 X2 Z3 X6 + +0.02058658 Z1 Y2 Z3 Y6 + +0.01111059 Z7 Y9 Y13 Z15 + -0.01111059 X9 Z10 X13 Z14 + +0.02166131 Z9 X10 Z11 X14 + +0.02166131 Z9 Y10 Z11 Y14 + +0.04519600 X16 Z17 X18 Z19 + +0.04519600 Y16 Z17 Y18 Z19 + +0.00793294 Z0 X1 Z3 X5 Z6 + -0.00793294 Z0 X1 Z4 X5 Z7 + -0.00793294 Y1 Z3 Z4 Y5 Z6 + +0.02058658 X2 Z3 Z5 X6 Z7 + +0.02058658 Y2 Z3 Z5 Y6 Z7 + +0.01111059 Z8 X9 Z11 X13 Z14 + -0.01111059 Y9 Z11 Z12 Y13 Z14 + +0.00793294 X0 Y1 X2 X4 Y5 X6 + +0.00793294 X0 Y1 X2 Y4 Y5 Y6 + -0.00793294 X0 Y1 Y2 X4 Y5 Y6 + +0.00793294 X0 Y1 Y2 Y4 Y5 X6 + +0.00793294 Y0 Y1 X2 X4 Y5 Y6 + -0.00793294 Y0 Y1 X2 Y4 Y5 X6 + +0.00793294 Y0 Y1 Y2 X4 Y5 X6 + +0.00793294 Y0 Y1 Y2 Y4 Y5 Y6 + -0.00793294 Z0 Y1 Z2 Z3 Y5 Z7 + +0.00793294 Z0 Y1 Z2 Z4 Y5 Z6 + +0.00793294 X1 Z2 Z3 Z4 X5 Z7 + -0.01111059 Z7 Z8 X9 Z12 X13 Z15 + +0.02166131 Z7 X10 Z11 Z13 X14 Z15 + +0.02166131 Z7 Y10 Z11 Z13 Y14 Z15 + +0.01111059 X8 Y9 X10 X12 Y13 X14 + +0.01111059 X8 Y9 X10 Y12 Y13 Y14 + -0.01111059 X8 Y9 Y10 X12 Y13 Y14 + +0.01111059 X8 Y9 Y10 Y12 Y13 X14 + +0.01111059 Y8 Y9 X10 X12 Y13 Y14 + -0.01111059 Y8 Y9 X10 Y12 Y13 X14 + +0.01111059 Y8 Y9 Y10 X12 Y13 X14 + +0.01111059 Y8 Y9 Y10 Y12 Y13 Y14 + +0.01111059 Z8 Y9 Z10 Z12 Y13 Z14 + -0.01111059 Z7 Z8 Y9 Z10 Z11 Y13 Z15 + +0.01111059 Z7 X9 Z10 Z11 Z12 X13 Z15
1,H_blue,T_57 + T_58 + T_59 + T_60 + T_73 + T_207 + T_332 + T_861 + T_1319 + T_1426 + T_1891 + T_2306 + T_2369 + T_2385 + T_2895 + T_2953 + T_3673 + T_3677 + T_3678 + T_3680,20,57,-2.4


=== Verify commuting groups after JW/BK mapping ===


,color_group,pair,fermionic_commute,JW_commute,BK_commute
0,red,"T_0, T_72",True,True,True
1,red,"T_0, T_190",True,True,True
2,red,"T_0, T_206",True,True,True
3,red,"T_0, T_849",True,True,True
4,red,"T_0, T_1318",True,True,True
...,...,...,...,...,...
25358,color_275,"T_2196, T_3327",True,True,True
25359,color_275,"T_3007, T_3256",True,True,True
25360,color_275,"T_3007, T_3327",True,True,True
25361,color_275,"T_3256, T_3327",True,True,True


In [7]:
# Find duplicated JW Pauli strings across fermionic terms T_i

from collections import defaultdict
import pandas as pd

pauli_usage = defaultdict(list)

for node in sorted(G.nodes()):
    JW_T = G.nodes[node]["JW_operator"]

    for pauli_key, coeff in JW_T.terms.items():
        pauli_string = format_qubit_term(pauli_key)

        pauli_usage[pauli_string].append(
            {
                "vertex": f"T_{node}",
                "coefficient": coeff,
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

duplicate_rows = []

for pauli_string, appearances in pauli_usage.items():
    if len(appearances) > 1:
        duplicate_rows.append(
            {
                "JW_Pauli_string": pauli_string,
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "coefficients": [x["coefficient"] for x in appearances],
            }
        )

duplicate_jw_df = pd.DataFrame(duplicate_rows)
duplicate_jw_df = duplicate_jw_df.sort_values(
    "number_of_appearances",
    ascending=False
).reset_index(drop=True)

print("Total JW Pauli-string appearances:", sum(len(G.nodes[node]["JW_operator"].terms) for node in G.nodes()))
print("Number of unique JW Pauli strings:", len(pauli_usage))
print("Number of duplicated JW Pauli strings:", len(duplicate_jw_df))

display(duplicate_jw_df)

Total JW Pauli-string appearances: 25441
Number of unique JW Pauli strings: 11691
Number of duplicated JW Pauli strings: 10061


,JW_Pauli_string,number_of_appearances,appears_in_vertices,coefficients
0,I,211,"[T_0, T_1, T_6, T_11, T_16, T_21, T_25, T_29, T_33, T_37, T_40, T_43, T_46, T_49, T_51, T_53, T_55, T_57, T_58, T_59, T_60, T_61, T_111, T_141, T_186, T_215, T_255, T_293, T_328, T_365, T_395, T_441, T_466, T_511, T_531, T_585, T_600, T_653, T_663, T_725, T_730, T_775, T_805, T_845, T_874, T_909, T_947, T_977, T_1014, T_1039, T_1085, T_1105, T_1150, T_1165, T_1219, T_1229, T_1282, T_1287, T_1308, T_1349, T_1373, T_1409, T_1433, T_1465, T_1496, T_1523, T_1554, T_1577, T_1615, T_1633, T_1671, T_1685, T_1730, T_1739, T_1784, T_1789, T_1825, T_1849, T_1881, T_1905, T_1932, T_1963, T_1986, T_2017, T_2035, T_2073, T_2087, T_2125, T_2134, T_2179, T_2184, T_2197, T_2229, T_2249, T_2277, T_2296, T_2320, T_2346, T_2366, T_2391, ...]","[13.76808794966473, -1.7679774994319637, -1.7679774994319637, -1.6555019362457821, -1.6555019362457821, -1.5630902062711896, -1.5630902062711896, -1.4664505063672386, -1.4664505063672386, -1.4107676907228812, -1.4107676907228812, -1.305051659990087, -1.305051659990087, -1.137441725788379, -1.137441725788379, -0.9816003783964337, -0.9816003783964337, -0.7964058499573899, -0.7964058499573899, -0.6673137423023049, -0.6673137423023049, 0.10191392055616116, 0.049391342399017824, 0.08223002261135531, 0.05728294465508936, 0.07957591868590091, 0.06114803594208264, 0.07724377089188787, 0.06878493495310864, 0.0836670461023376, 0.07252641022137937, 0.08527729313134724, 0.0726800491728012, 0.08205023079631743, 0.07795149507488103, 0.08674332134356735, 0.08212526725185146, 0.08988142709584254, 0.10140180050126363, 0.11133560889472865, 0.08223002261135531, 0.049391342399017824, 0.07957591868590091, 0.05728294465508936, 0.07724377089188787, 0.06114803594208264, 0.0836670461023376, 0.06878493495310864, 0.08527729313134724, 0.07252641022137937, 0.08205023079631743, 0.0726800491728012, 0.08674332134356735, 0.07795149507488103, 0.08988142709584254, 0.08212526725185146, 0.11133560889472865, 0.10140180050126363, 0.08766914141799742, 0.05027721373846579, 0.07790091878059768, 0.057277549630481664, 0.07786412825339058, 0.06340872425214235, 0.07983834944104189, 0.06768235920657868, 0.08118583198191065, 0.0703640306642336, 0.08182610432894313, 0.07440969213402784, 0.08373025748616564, 0.08314160859890551, 0.09373855541314824, 0.08294669631524386, 0.09027954513132128, 0.07790091878059768, 0.05027721373846579, 0.07786412825339058, 0.057277549630481664, 0.07983834944104189, 0.06340872425214235, 0.08118583198191065, 0.06768235920657868, 0.08182610432894313, 0.0703640306642336, 0.08373025748616564, 0.07440969213402784, 0.09373855541314824, 0.08314160859890551, 0.09027954513132128, 0.08294669631524386, 0.08285169886065757, 0.050657167395117, 0.0773669648739134, 0.05839484379620484, 0.07885605540201591, 0.0643686821298091, 0.07997402315436834, 0.0676968610754815, 0.08080761110215218, ...]"
1,Z11,20,"[T_46, T_441, T_1039, T_1554, T_1986, T_2346, T_2643, T_2886, T_3084, T_3237, T_3359, T_3450, T_3527, T_3537, T_3545, T_3553, T_3561, T_3566, T_3577, T_3580]","[1.305051659990087, -0.08527729313134724, -0.07252641022137937, -0.08118583198191065, -0.06768235920657868, -0.07997402315436834, -0.0643686821298091, -0.07948347741548079, -0.06090134206715498, -0.0826178707997248, -0.06063496885970733, -0.0859051639186294, -0.08351755713983326, -0.05515552756086485, -0.08634889311814015, -0.0646875880322408, -0.08942345882819708, -0.07170233442860685, -0.09557970288804435, -0.07946875903458667]"
2,Z4,20,"[T_21, T_186, T_805, T_1349, T_1789, T_2197, T_2229, T_2249, T_2277, T_2296, T_2320, T_2346, T_2366, T_2391, T_2407, T_2439, T_2451, T_2482, T_2490, T_2528]","[1.5630902062711896, -0.05728294465508936, -0.07957591868590091, -0.05027721373846579, -0.07790091878059768, -0.08285169886065757, -0.050657167395117, -0.0773669648739134, -0.05839484379620484, -0.07885605540201591, -0.0643686821298091, -0.07997402315436834, -0.0676968610754815, -0.08080761110215218, -0.073387498545

In [8]:
# Compute Pauli-duplication ratio for JW and BK

from collections import defaultdict
import pandas as pd

from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner, bravyi_kitaev, normal_ordered


def pauli_support(qubit_op, tol=1e-12, include_identity=True):
    """
    Return the set of Pauli strings with nonzero coefficients.
    """
    qubit_op.compress(abs_tol=tol)

    support = set()

    for pauli_key, coeff in qubit_op.terms.items():
        if abs(coeff) <= tol:
            continue

        if not include_identity and pauli_key == ():
            continue

        support.add(pauli_key)

    return support


def map_fermion_to_qubit(op, mapping="JW", n_qubits=None):
    """
    Map a FermionOperator to a QubitOperator using JW or BK.
    """
    mapping = mapping.upper()

    if mapping == "JW":
        qop = jordan_wigner(op)

    elif mapping == "BK":
        if n_qubits is None:
            raise ValueError("n_qubits is required for BK.")

        try:
            qop = bravyi_kitaev(op, n_qubits=n_qubits)
        except TypeError:
            qop = bravyi_kitaev(op, n_qubits)

    else:
        raise ValueError("mapping must be 'JW' or 'BK'.")

    qop.compress(abs_tol=1e-12)
    return qop


def pauli_duplication_ratio(
    fermionic_terms,
    mapping="JW",
    n_qubits=None,
    include_identity=True,
    tol=1e-12,
):
    """
    Compute

        sum_alpha #mapping(H_alpha) / #mapping(H)

    where H = sum_alpha H_alpha.

    Also returns a duplicate-use table.
    """

    H_full = FermionOperator.zero()

    numerator = 0
    union_support = set()
    pauli_usage = defaultdict(list)

    for alpha, H_alpha in enumerate(fermionic_terms):
        H_full += H_alpha

        Q_alpha = map_fermion_to_qubit(
            H_alpha,
            mapping=mapping,
            n_qubits=n_qubits,
        )

        support_alpha = pauli_support(
            Q_alpha,
            tol=tol,
            include_identity=include_identity,
        )

        numerator += len(support_alpha)
        union_support |= support_alpha

        for pauli_key, coeff in Q_alpha.terms.items():
            if abs(coeff) <= tol:
                continue

            if not include_identity and pauli_key == ():
                continue

            pauli_usage[pauli_key].append(
                {
                    "vertex": f"T_{alpha}",
                    "coefficient": coeff,
                }
            )

    H_full = normal_ordered(H_full)
    H_full.compress(abs_tol=tol)

    Q_full = map_fermion_to_qubit(
        H_full,
        mapping=mapping,
        n_qubits=n_qubits,
    )

    full_support = pauli_support(
        Q_full,
        tol=tol,
        include_identity=include_identity,
    )

    denominator = len(full_support)

    duplication_ratio = numerator / denominator
    union_ratio = numerator / len(union_support)

    summary_df = pd.DataFrame(
        [
            {
                "mapping": mapping.upper(),
                "include_identity": include_identity,
                "sum_alpha_number_of_Pauli_strings": numerator,
                "number_of_unique_Pauli_strings_before_cancellation": len(union_support),
                "number_of_Pauli_strings_in_full_H": denominator,
                "duplication_ratio": duplication_ratio,
                "raw_reuse_ratio_before_cancellation": union_ratio,
            }
        ]
    )

    duplicate_rows = []

    for pauli_key, appearances in pauli_usage.items():
        if len(appearances) <= 1:
            continue

        combined_coeff = Q_full.terms.get(pauli_key, 0.0)

        duplicate_rows.append(
            {
                "Pauli_string": format_qubit_term(pauli_key),
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "individual_coefficients": [complex(x["coefficient"]) for x in appearances],
                "combined_coefficient_in_full_H": complex(combined_coeff),
                "survives_in_full_H": abs(combined_coeff) > tol,
            }
        )

    duplicate_df = pd.DataFrame(duplicate_rows)

    if len(duplicate_df) > 0:
        duplicate_df = duplicate_df.sort_values(
            "number_of_appearances",
            ascending=False,
        ).reset_index(drop=True)

    return summary_df, duplicate_df


# ------------------------------------------------------------
# Run for JW
# ------------------------------------------------------------

n_qubits = molecule.n_qubits

jw_summary_df, jw_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="JW",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== JW Pauli-duplication ratio ===")
display(jw_summary_df)

print("\n=== Duplicated JW Pauli strings ===")
display(jw_duplicate_df)


# ------------------------------------------------------------
# Optional: run for BK too
# ------------------------------------------------------------

bk_summary_df, bk_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="BK",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== BK Pauli-duplication ratio ===")
display(bk_summary_df)

print("\n=== Duplicated BK Pauli strings ===")
display(bk_duplicate_df)

=== JW Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,JW,True,25441,11691,7151,3.557684,2.176118



=== Duplicated JW Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,211,"[T_0, T_1, T_6, T_11, T_16, T_21, T_25, T_29, T_33, T_37, T_40, T_43, T_46, T_49, T_51, T_53, T_55, T_57, T_58, T_59, T_60, T_61, T_111, T_141, T_186, T_215, T_255, T_293, T_328, T_365, T_395, T_441, T_466, T_511, T_531, T_585, T_600, T_653, T_663, T_725, T_730, T_775, T_805, T_845, T_874, T_909, T_947, T_977, T_1014, T_1039, T_1085, T_1105, T_1150, T_1165, T_1219, T_1229, T_1282, T_1287, T_1308, T_1349, T_1373, T_1409, T_1433, T_1465, T_1496, T_1523, T_1554, T_1577, T_1615, T_1633, T_1671, T_1685, T_1730, T_1739, T_1784, T_1789, T_1825, T_1849, T_1881, T_1905, T_1932, T_1963, T_1986, T_2017, T_2035, T_2073, T_2087, T_2125, T_2134, T_2179, T_2184, T_2197, T_2229, T_2249, T_2277, T_2296, T_2320, T_2346, T_2366, T_2391, ...]","[(13.76808794966473+0j), (-1.7679774994319637+0j), (-1.7679774994319637+0j), (-1.6555019362457821+0j), (-1.6555019362457821+0j), (-1.5630902062711896+0j), (-1.5630902062711896+0j), (-1.4664505063672386+0j), (-1.4664505063672386+0j), (-1.4107676907228812+0j), (-1.4107676907228812+0j), (-1.305051659990087+0j), (-1.305051659990087+0j), (-1.137441725788379+0j), (-1.137441725788379+0j), (-0.9816003783964337+0j), (-0.9816003783964337+0j), (-0.7964058499573899+0j), (-0.7964058499573899+0j), (-0.6673137423023049+0j), (-0.6673137423023049+0j), (0.10191392055616116+0j), (0.049391342399017824+0j), (0.08223002261135531+0j), (0.05728294465508936+0j), (0.07957591868590091+0j), (0.06114803594208264+0j), (0.07724377089188787+0j), (0.06878493495310864+0j), (0.0836670461023376+0j), (0.07252641022137937+0j), (0.08527729313134724+0j), (0.0726800491728012+0j), (0.08205023079631743+0j), (0.07795149507488103+0j), (0.08674332134356735+0j), (0.08212526725185146+0j), (0.08988142709584254+0j), (0.10140180050126363+0j), (0.11133560889472865+0j), (0.08223002261135531+0j), (0.049391342399017824+0j), (0.07957591868590091+0j), (0.05728294465508936+0j), (0.07724377089188787+0j), (0.06114803594208264+0j), (0.0836670461023376+0j), (0.06878493495310864+0j), (0.08527729313134724+0j), (0.07252641022137937+0j), (0.08205023079631743+0j), (0.0726800491728012+0j), (0.08674332134356735+0j), (0.07795149507488103+0j), (0.08988142709584254+0j), (0.08212526725185146+0j), (0.11133560889472865+0j), (0.10140180050126363+0j), (0.08766914141799742+0j), (0.05027721373846579+0j), (0.07790091878059768+0j), (0.057277549630481664+0j), (0.07786412825339058+0j), (0.06340872425214235+0j), (0.07983834944104189+0j), (0.06768235920657868+0j), (0.08118583198191065+0j), (0.0703640306642336+0j), (0.08182610432894313+0j), (0.07440969213402784+0j), (0.08373025748616564+0j), (0.08314160859890551+0j), (0.09373855541314824+0j), (0.08294669631524386+0j), (0.09027954513132128+0j), (0.07790091878059768+0j), (0.05027721373846579+0j), (0.07786412825339058+0j), (0.057277549630481664+0j), (0.07983834944104189+0j), (0.06340872425214235+0j), (0.08118583198191065+0j), (0.06768235920657868+0j), (0.08182610432894313+0j), (0.0703640306642336+0j), (0.08373025748616564+0j), (0.07440969213402784+0j), (0.09373855541314824+0j), (0.08314160859890551+0j), (0.09027954513132128+0j), (0.08294669631524386+0j), (0.08285169886065757+0j), (0.050657167395117+0j), (0.0773669648739134+0j), (0.05839484379620484+0j), (0.07885605540201591+0j), (0.0643686821298091+0j), (0.07997402315436834+0j), (0.0676968610754815+0j), (0.08080761110215218+0j), ...]",3.060064+0.000000j,True
1,Z11,20,"[T_46, T_441, T_1039, T_1554, T_1986, T_2346, T_2643, T_2886, T_3084, T_3237, T_3359, T_3450, T_3527, T_3537, T_3545, T_3553, T_3561, T_3566, T_3577, T_3580]","[(1.305051659990087+0j), (-0.08527729313134724+0j), (-0.07252641022137937+0j), (-0.08118583198191065+0j), (-0.06768235920657868+0j), (-0.07997402315436834+0j), (-0.0643686821298091+0j), (-0.07948347741548079+0j), (-0.06090134206715498+0j), (-0.0826178707997248+0j), (-0.06063496885970733+0j), (-0.0859051639186294+0j), (-0.083517557139833

=== BK Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,BK,True,25441,11691,7151,3.557684,2.176118



=== Duplicated BK Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,211,"[T_0, T_1, T_6, T_11, T_16, T_21, T_25, T_29, T_33, T_37, T_40, T_43, T_46, T_49, T_51, T_53, T_55, T_57, T_58, T_59, T_60, T_61, T_111, T_141, T_186, T_215, T_255, T_293, T_328, T_365, T_395, T_441, T_466, T_511, T_531, T_585, T_600, T_653, T_663, T_725, T_730, T_775, T_805, T_845, T_874, T_909, T_947, T_977, T_1014, T_1039, T_1085, T_1105, T_1150, T_1165, T_1219, T_1229, T_1282, T_1287, T_1308, T_1349, T_1373, T_1409, T_1433, T_1465, T_1496, T_1523, T_1554, T_1577, T_1615, T_1633, T_1671, T_1685, T_1730, T_1739, T_1784, T_1789, T_1825, T_1849, T_1881, T_1905, T_1932, T_1963, T_1986, T_2017, T_2035, T_2073, T_2087, T_2125, T_2134, T_2179, T_2184, T_2197, T_2229, T_2249, T_2277, T_2296, T_2320, T_2346, T_2366, T_2391, ...]","[(13.76808794966473+0j), (-1.7679774994319637+0j), (-1.7679774994319637+0j), (-1.6555019362457821+0j), (-1.6555019362457821+0j), (-1.5630902062711896+0j), (-1.5630902062711896+0j), (-1.4664505063672386+0j), (-1.4664505063672386+0j), (-1.4107676907228812+0j), (-1.4107676907228812+0j), (-1.305051659990087+0j), (-1.305051659990087+0j), (-1.137441725788379+0j), (-1.137441725788379+0j), (-0.9816003783964337+0j), (-0.9816003783964337+0j), (-0.7964058499573899+0j), (-0.7964058499573899+0j), (-0.6673137423023049+0j), (-0.6673137423023049+0j), (0.10191392055616116+0j), (0.049391342399017824+0j), (0.08223002261135531+0j), (0.05728294465508936+0j), (0.07957591868590091+0j), (0.06114803594208264+0j), (0.07724377089188787+0j), (0.06878493495310864+0j), (0.0836670461023376+0j), (0.07252641022137937+0j), (0.08527729313134724+0j), (0.0726800491728012+0j), (0.08205023079631743+0j), (0.07795149507488103+0j), (0.08674332134356735+0j), (0.08212526725185146+0j), (0.08988142709584254+0j), (0.10140180050126363+0j), (0.11133560889472865+0j), (0.08223002261135531+0j), (0.049391342399017824+0j), (0.07957591868590091+0j), (0.05728294465508936+0j), (0.07724377089188787+0j), (0.06114803594208264+0j), (0.0836670461023376+0j), (0.06878493495310864+0j), (0.08527729313134724+0j), (0.07252641022137937+0j), (0.08205023079631743+0j), (0.0726800491728012+0j), (0.08674332134356735+0j), (0.07795149507488103+0j), (0.08988142709584254+0j), (0.08212526725185146+0j), (0.11133560889472865+0j), (0.10140180050126363+0j), (0.08766914141799742+0j), (0.05027721373846579+0j), (0.07790091878059768+0j), (0.057277549630481664+0j), (0.07786412825339058+0j), (0.06340872425214235+0j), (0.07983834944104189+0j), (0.06768235920657868+0j), (0.08118583198191065+0j), (0.0703640306642336+0j), (0.08182610432894313+0j), (0.07440969213402784+0j), (0.08373025748616564+0j), (0.08314160859890551+0j), (0.09373855541314824+0j), (0.08294669631524386+0j), (0.09027954513132128+0j), (0.07790091878059768+0j), (0.05027721373846579+0j), (0.07786412825339058+0j), (0.057277549630481664+0j), (0.07983834944104189+0j), (0.06340872425214235+0j), (0.08118583198191065+0j), (0.06768235920657868+0j), (0.08182610432894313+0j), (0.0703640306642336+0j), (0.08373025748616564+0j), (0.07440969213402784+0j), (0.09373855541314824+0j), (0.08314160859890551+0j), (0.09027954513132128+0j), (0.08294669631524386+0j), (0.08285169886065757+0j), (0.050657167395117+0j), (0.0773669648739134+0j), (0.05839484379620484+0j), (0.07885605540201591+0j), (0.0643686821298091+0j), (0.07997402315436834+0j), (0.0676968610754815+0j), (0.08080761110215218+0j), ...]",3.060064+0.000000j,True
1,Z9 Z10 Z11,20,"[T_46, T_441, T_1039, T_1554, T_1986, T_2346, T_2643, T_2886, T_3084, T_3237, T_3359, T_3450, T_3527, T_3537, T_3545, T_3553, T_3561, T_3566, T_3577, T_3580]","[(1.305051659990087+0j), (-0.08527729313134724+0j), (-0.07252641022137937+0j), (-0.08118583198191065+0j), (-0.06768235920657868+0j), (-0.07997402315436834+0j), (-0.0643686821298091+0j), (-0.07948347741548079+0j), (-0.06090134206715498+0j), (-0.0826178707997248+0j), (-0.06063496885970733+0j), (-0.0859051639186294+0j), (-0.08351755